In [ ]:
"""Day 25 / Session 3 - Lab Assignment 1: The Masked-LM Objective - 80/10/10 Masking Pipeline & MLM Cross-Entropy."""
import numpy as np


def maskable_positions(tokens, special_ids=()):
    special = set(special_ids)
    return [i for i, t in enumerate(tokens) if t not in special]          # left to right


def choose_mask_positions(tokens, mask_prob=0.15, special_ids=(), seed=0):
    candidates = maskable_positions(tokens, special_ids)
    n = len(candidates)
    if n == 0:
        return []
    k = min(n, max(1, int(round(mask_prob * n))))
    rng = np.random.RandomState(seed)
    chosen = rng.choice(candidates, size=k, replace=False)
    return sorted(int(p) for p in chosen)


def apply_masking(tokens, positions, mask_id, vocab_size, seed=0):
    masked = [int(t) for t in tokens]
    labels = [-100] * len(masked)
    rng = np.random.RandomState(seed)
    for p in sorted(positions):
        labels[p] = int(tokens[p])                                         # the original token is the target
        r = rng.rand()
        if r < 0.8:
            masked[p] = mask_id                                            # 80%: [MASK]
        elif r < 0.9:
            masked[p] = int(rng.randint(0, vocab_size))                    # 10%: a random token
        # else: 10% unchanged
    return masked, labels


def mlm_cross_entropy(logits, labels, ignore_index=-100):
    logits = np.asarray(logits, dtype=float)
    labels = np.asarray(labels)
    keep = labels != ignore_index
    if not keep.any():
        return 0.0
    z = logits[keep]
    z = z - z.max(axis=-1, keepdims=True)
    log_probs = z - np.log(np.exp(z).sum(axis=-1, keepdims=True))          # stable log-softmax
    return float(-np.mean(log_probs[np.arange(len(z)), labels[keep]]))
